**Step 1 | Install and initialize PySpark**

In [1]:
!pip -q install pyspark


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
import os

spark = SparkSession.builder.appName("LogisticsPracticeSet02").getOrCreate()


**Step 2 | Upload the dataset files**

In [3]:
from google.colab import files
uploaded = files.upload()


Saving carrier_master.csv to carrier_master.csv
Saving route_master.csv to route_master.csv
Saving shipments_75000.csv to shipments_75000.csv
Saving shipments_extra_12000.csv to shipments_extra_12000.csv
Saving warehouse_master.csv to warehouse_master.csv
Saving warehouse_sensor_config.json to warehouse_sensor_config.json


**Step 3 | Create the DataFrames**

In [4]:
shipments_df = spark.read.option("header", True).option("inferSchema", True).csv("shipments_75000.csv")
extra_df = spark.read.option("header", True).option("inferSchema", True).csv("shipments_extra_12000.csv")
warehouse_df = spark.read.option("header", True).option("inferSchema", True).csv("warehouse_master.csv")
carrier_df = spark.read.option("header", True).option("inferSchema", True).csv("carrier_master.csv")
route_df = spark.read.option("header", True).option("inferSchema", True).csv("route_master.csv")
sensor_df = spark.read.option("multiline", True).option("inferSchema", True).json("warehouse_sensor_config.json")

shipments_df.printSchema()
shipments_df.show(5, truncate=False)
warehouse_df.show(5, truncate=False)
carrier_df.show(5, truncate=False)
route_df.show(5, truncate=False)
sensor_df.show(5, truncate=False)


root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

+-----------+------------+-------------+----------------------+------------

# **Practice Exercises**

**Part A | DataFrame Operations**

In [5]:
# 1. Display the schema, total number of records, and first 15 shipments.
shipments_df.printSchema()
print("Total shipments:", shipments_df.count())
shipments_df.show(15, truncate=False)


root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

Total shipments: 75000
+-----------+------------+-------------+------------

In [6]:
# 2. Select the required shipment columns.
shipments_df.select(
    "shipment_id", "warehouse_id", "service_level",
    "shipment_status", "shipping_fee_inr"
).show(10, truncate=False)


+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
|SHP0000001 |W08         |Express      |Delivered      |554.34          |
|SHP0000002 |W08         |Standard     |Delivered      |414.56          |
|SHP0000003 |W01         |Express      |Delivered      |724.64          |
|SHP0000004 |W03         |Standard     |Delivered      |287.86          |
|SHP0000005 |W03         |Economy      |Delivered      |251.68          |
|SHP0000006 |W01         |Express      |Delivered      |477.9           |
|SHP0000007 |W02         |Standard     |In Transit     |298.68          |
|SHP0000008 |W03         |Economy      |In Transit     |284.52          |
|SHP0000009 |W02         |Standard     |Delivered      |861.74          |
|SHP0000010 |W02         |Economy      |Delivered      |309.76          |
+-----------+------------+------------

In [7]:
# 3. Filter Express shipments where package weight is greater than 5 kg.
express_heavy_df = shipments_df.filter(
    (F.col("service_level") == "Express") &
    (F.col("package_weight_kg") > 5)
)
express_heavy_df.show(10, truncate=False)


+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0000003 |2026-01-31  |2026-02-01   |2026-02-03            |2

In [8]:
# 4. Sort by shipping fee in descending order and display the top 20.
shipments_df.orderBy(F.col("shipping_fee_inr").desc()).show(20, truncate=False)


+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0044904 |2026-08-11  |2026-08-13   |2026-08-16            |N

**Part B | Aggregations and Joins**

In [9]:
# 5. Count shipments for each shipment status.
shipments_df.groupBy("shipment_status").count().show()


+---------------+-----+
|shipment_status|count|
+---------------+-----+
|       Returned| 6130|
|           Lost| 2241|
|     In Transit|10472|
|      Cancelled| 3723|
|      Delivered|52434|
+---------------+-----+



In [10]:
# 6. Calculate total fees, total costs, and profit margin for each service level.
service_summary_df = shipments_df.groupBy("service_level").agg(
    F.sum("shipping_fee_inr").alias("total_shipping_fees"),
    F.sum("shipping_cost_inr").alias("total_shipping_costs")
).withColumn(
    "profit_margin_pct",
    F.when(F.col("total_shipping_fees") != 0,
           (F.col("total_shipping_fees") - F.col("total_shipping_costs")) /
           F.col("total_shipping_fees") * 100)
)
service_summary_df.show(truncate=False)


+-------------+--------------------+--------------------+------------------+
|service_level|total_shipping_fees |total_shipping_costs|profit_margin_pct |
+-------------+--------------------+--------------------+------------------+
|Express      |1.0058468059999986E7|7350805.699999983   |26.919232072403737|
|Standard     |1.3437372619999977E7|1.0110465900000025E7|24.758610288504137|
|Economy      |4489918.680000002   |3386212.500000007   |24.581874609809063|
+-------------+--------------------+--------------------+------------------+



In [11]:
# 7. Join shipments with warehouse details and count shipments by region.
shipment_region_df = shipments_df.join(warehouse_df, on="warehouse_id", how="left")
shipment_region_df.groupBy("region").agg(
    F.count("shipment_id").alias("shipment_count")
).show()


+------+--------------+
|region|shipment_count|
+------+--------------+
| South|         33018|
|  East|          5306|
|  West|         26909|
| North|          9767|
+------+--------------+



In [12]:
# 8. Join shipments, carriers, and routes. Find average shipping cost per km by carrier.
carrier_route_df = shipments_df.join(carrier_df, on="carrier_id", how="left").join(
    route_df, on="route_id", how="left"
)
carrier_route_df.filter(
    F.col("shipping_cost_inr").isNotNull() &
    F.col("distance_km").isNotNull() &
    (F.col("distance_km") > 0)
).withColumn(
    "cost_per_km", F.col("shipping_cost_inr") / F.col("distance_km")
).groupBy("carrier_id", "carrier_name").agg(
    F.avg("cost_per_km").alias("avg_shipping_cost_per_km")
).show(truncate=False)


+----------+--------------------+------------------------+
|carrier_id|carrier_name        |avg_shipping_cost_per_km|
+----------+--------------------+------------------------+
|C07       |BluePeak Delivery   |0.6074832283099014      |
|NULL      |NULL                |0.6318401277501529      |
|C04       |AirSprint Couriers  |0.6125755103088726      |
|C06       |RouteWise           |0.5879324060720517      |
|C03       |MetroRoute Logistics|0.594275960971091       |
|C02       |ParcelBridge        |0.5957409571811839      |
|C05       |CargoNest           |0.5897283901696387      |
|C01       |SwiftLine Express   |0.6032748764541479      |
+----------+--------------------+------------------------+



**Part C | Null Handling and Conditional Logic**

In [13]:
# 9. Count missing values in the selected columns.
columns_to_check = ["carrier_id", "package_weight_kg", "shipping_cost_inr", "customer_rating"]
null_counts = shipments_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls")
    for c in columns_to_check
])
null_counts.show()


+----------------+-----------------------+-----------------------+---------------------+
|carrier_id_nulls|package_weight_kg_nulls|shipping_cost_inr_nulls|customer_rating_nulls|
+----------------+-----------------------+-----------------------+---------------------+
|             871|                   1129|                   1503|                26255|
+----------------+-----------------------+-----------------------+---------------------+



In [14]:
# 10. Fill missing handling minutes with zero in one DataFrame.
handling_filled_df = shipments_df.fillna({"handling_minutes": 0})
handling_filled_df.select("shipment_id", "handling_minutes").show(10)

# Create another DataFrame without missing carrier IDs.
known_carrier_df = shipments_df.filter(F.col("carrier_id").isNotNull())
print("Shipments with carrier ID:", known_carrier_df.count())


+-----------+----------------+
|shipment_id|handling_minutes|
+-----------+----------------+
| SHP0000001|           122.0|
| SHP0000002|           220.0|
| SHP0000003|           188.0|
| SHP0000004|           179.0|
| SHP0000005|           218.0|
| SHP0000006|            48.0|
| SHP0000007|           214.0|
| SHP0000008|            76.0|
| SHP0000009|           222.0|
| SHP0000010|            98.0|
+-----------+----------------+
only showing top 10 rows
Shipments with carrier ID: 74129


In [15]:
# 11. Create weight_category using when() and otherwise().
weight_category_df = shipments_df.withColumn(
    "weight_category",
    F.when(F.col("package_weight_kg").isNull(), "Unknown")
     .when(F.col("package_weight_kg") < 2, "Light")
     .when(F.col("package_weight_kg") <= 10, "Medium")
     .otherwise("Heavy")
)
weight_category_df.select("shipment_id", "package_weight_kg", "weight_category").show(10)


+-----------+-----------------+---------------+
|shipment_id|package_weight_kg|weight_category|
+-----------+-----------------+---------------+
| SHP0000001|             3.93|         Medium|
| SHP0000002|             5.62|         Medium|
| SHP0000003|             8.78|         Medium|
| SHP0000004|             0.97|          Light|
| SHP0000005|             NULL|        Unknown|
| SHP0000006|             6.05|         Medium|
| SHP0000007|             1.36|          Light|
| SHP0000008|             2.04|         Medium|
| SHP0000009|            18.73|          Heavy|
| SHP0000010|             2.02|         Medium|
+-----------+-----------------+---------------+
only showing top 10 rows


**Part D | Date and Time Functions**

In [16]:
# 12. Convert delivery dates to date type and calculate delivery duration for delivered shipments.
dates_df = shipments_df.withColumn("booking_date", F.to_date("booking_date")) \
    .withColumn("promised_delivery_date", F.to_date("promised_delivery_date")) \
    .withColumn("actual_delivery_date", F.to_date("actual_delivery_date"))

delivered_df = dates_df.filter(F.col("shipment_status") == "Delivered").withColumn(
    "delivery_duration_days",
    F.datediff(F.col("actual_delivery_date"), F.col("booking_date"))
)
delivered_df.select("shipment_id", "booking_date", "actual_delivery_date", "delivery_duration_days").show(10)


+-----------+------------+--------------------+----------------------+
|shipment_id|booking_date|actual_delivery_date|delivery_duration_days|
+-----------+------------+--------------------+----------------------+
| SHP0000001|  2026-05-24|          2026-05-27|                     3|
| SHP0000002|  2026-05-17|          2026-05-21|                     4|
| SHP0000003|  2026-01-31|          2026-02-04|                     4|
| SHP0000004|  2026-03-12|          2026-03-18|                     6|
| SHP0000005|  2026-08-16|          2026-08-29|                    13|
| SHP0000006|  2026-02-06|          2026-02-09|                     3|
| SHP0000009|  2026-08-12|          2026-08-21|                     9|
| SHP0000010|  2026-06-01|          2026-06-11|                    10|
| SHP0000011|  2026-04-24|          2026-05-01|                     7|
| SHP0000013|  2026-06-26|          2026-06-29|                     3|
+-----------+------------+--------------------+----------------------+
only s

In [17]:
# 13. Classify delivery performance using actual and promised delivery dates.
performance_df = dates_df.withColumn(
    "delivery_performance",
    F.when(F.col("actual_delivery_date").isNull(), "Not Delivered / Missing Actual Date")
     .when(F.col("actual_delivery_date") < F.col("promised_delivery_date"), "Early")
     .when(F.col("actual_delivery_date") == F.col("promised_delivery_date"), "On Time")
     .otherwise("Late")
)
performance_df.select("shipment_id", "promised_delivery_date", "actual_delivery_date", "delivery_performance").show(10)


+-----------+----------------------+--------------------+--------------------+
|shipment_id|promised_delivery_date|actual_delivery_date|delivery_performance|
+-----------+----------------------+--------------------+--------------------+
| SHP0000001|            2026-05-26|          2026-05-27|                Late|
| SHP0000002|            2026-05-21|          2026-05-21|             On Time|
| SHP0000003|            2026-02-03|          2026-02-04|                Late|
| SHP0000004|            2026-03-17|          2026-03-18|                Late|
| SHP0000005|            2026-08-25|          2026-08-29|                Late|
| SHP0000006|            2026-02-09|          2026-02-09|             On Time|
| SHP0000007|            2026-01-14|                NULL|Not Delivered / M...|
| SHP0000008|            2026-03-16|                NULL|Not Delivered / M...|
| SHP0000009|            2026-08-18|          2026-08-21|                Late|
| SHP0000010|            2026-06-10|          2026-0

In [18]:
# 14. Monthly delivered-shipment counts and total shipping fees based on booking_date.
monthly_df = dates_df.filter(F.col("shipment_status") == "Delivered").withColumn(
    "booking_month", F.date_format("booking_date", "yyyy-MM")
).groupBy("booking_month").agg(
    F.count("shipment_id").alias("delivered_shipment_count"),
    F.sum("shipping_fee_inr").alias("total_shipping_fees")
).orderBy("booking_month")
monthly_df.show(truncate=False)


+-------------+------------------------+-------------------+
|booking_month|delivered_shipment_count|total_shipping_fees|
+-------------+------------------------+-------------------+
|2026-01      |6710                    |2525138.440000005  |
|2026-02      |6097                    |2268113.5000000037 |
|2026-03      |6810                    |2530456.2400000026 |
|2026-04      |6379                    |2389163.4600000014 |
|2026-05      |6858                    |2549683.9999999963 |
|2026-06      |6615                    |2447590.1200000006 |
|2026-07      |6773                    |2512355.2199999997 |
|2026-08      |6192                    |2306432.5199999996 |
+-------------+------------------------+-------------------+



**Part E | Window Functions**

In [19]:
# 15. Rank carriers by total shipping fees within each warehouse region using dense_rank().
region_carrier_df = shipment_region_df.join(carrier_df, on="carrier_id", how="left").groupBy(
    "region", "carrier_id", "carrier_name"
).agg(F.sum("shipping_fee_inr").alias("total_shipping_fees"))
region_window = Window.partitionBy("region").orderBy(F.col("total_shipping_fees").desc())
region_carrier_df.withColumn("carrier_rank", F.dense_rank().over(region_window)).orderBy(
    "region", "carrier_rank"
).show(truncate=False)


+------+----------+--------------------+-------------------+------------+
|region|carrier_id|carrier_name        |total_shipping_fees|carrier_rank|
+------+----------+--------------------+-------------------+------------+
|East  |C01       |SwiftLine Express   |393862.61999999994 |1           |
|East  |C02       |ParcelBridge        |373590.02          |2           |
|East  |C03       |MetroRoute Logistics|302260.92000000004 |3           |
|East  |C05       |CargoNest           |243322.91999999998 |4           |
|East  |C07       |BluePeak Delivery   |222025.85999999993 |5           |
|East  |C04       |AirSprint Couriers  |211583.64000000004 |6           |
|East  |C06       |RouteWise           |195810.3399999999  |7           |
|East  |NULL      |NULL                |24594.6            |8           |
|North |C02       |ParcelBridge        |731442.0999999994  |1           |
|North |C01       |SwiftLine Express   |653223.6200000005  |2           |
|North |C03       |MetroRoute Logistic

In [20]:
# 16. Find the latest booked shipment for each warehouse using row_number().
latest_window = Window.partitionBy("warehouse_id").orderBy(
    F.to_date("booking_date").desc(), F.col("shipment_id").desc()
)
latest_shipment_df = shipments_df.withColumn(
    "row_num", F.row_number().over(latest_window)
).filter(F.col("row_num") == 1)
latest_shipment_df.show(truncate=False)


+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+-------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|row_num|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+-------+
|SHP0072346 |2026-08-28  |NULL         |

In [21]:
# 17. Calculate a running total of shipping fees by warehouse, ordered by booking date and shipment ID.
running_window = Window.partitionBy("warehouse_id").orderBy(
    F.to_date("booking_date"), "shipment_id"
).rowsBetween(Window.unboundedPreceding, Window.currentRow)
running_df = shipments_df.withColumn(
    "running_shipping_fees", F.sum("shipping_fee_inr").over(running_window)
)
running_df.select("warehouse_id", "booking_date", "shipment_id", "shipping_fee_inr", "running_shipping_fees").show(10)


+------------+------------+-----------+----------------+---------------------+
|warehouse_id|booking_date|shipment_id|shipping_fee_inr|running_shipping_fees|
+------------+------------+-----------+----------------+---------------------+
|         W01|  2026-01-01| SHP0004062|          312.48|               312.48|
|         W01|  2026-01-01| SHP0006739|          227.02|                539.5|
|         W01|  2026-01-01| SHP0007553|          316.02|               855.52|
|         W01|  2026-01-01| SHP0008505|          572.68|   1428.1999999999998|
|         W01|  2026-01-01| SHP0009674|           322.9|               1751.1|
|         W01|  2026-01-01| SHP0011312|           390.1|               2141.2|
|         W01|  2026-01-01| SHP0011588|          421.72|              2562.92|
|         W01|  2026-01-01| SHP0011698|          190.22|              2753.14|
|         W01|  2026-01-01| SHP0011973|          384.64|   3137.7799999999997|
|         W01|  2026-01-01| SHP0017305|          203

**Part F | Union and Complex JSON**

In [22]:
# 18. Combine the main and extra shipment DataFrames using unionByName().
combined_df = shipments_df.unionByName(extra_df)
print("Main records:", shipments_df.count())
print("Extra records:", extra_df.count())
print("Combined records:", combined_df.count())


Main records: 75000
Extra records: 12000
Combined records: 87000


In [23]:
# 19. Find repeated shipment IDs and remove duplicate shipment records.
duplicate_ids_df = combined_df.groupBy("shipment_id").count().filter(F.col("count") > 1)
duplicate_ids_df.show(truncate=False)
print("Rows before deduplication:", combined_df.count())
deduplicated_df = combined_df.dropDuplicates(["shipment_id"])
print("Rows after deduplication:", deduplicated_df.count())


+-----------+-----+
|shipment_id|count|
+-----------+-----+
|SHP0039848 |2    |
|SHP0048411 |2    |
|SHP0052790 |2    |
|SHP0026413 |2    |
|SHP0041611 |2    |
|SHP0010090 |2    |
|SHP0038340 |2    |
|SHP0027530 |2    |
|SHP0031755 |2    |
|SHP0034259 |2    |
|SHP0047068 |2    |
|SHP0048280 |2    |
|SHP0008086 |2    |
|SHP0031960 |2    |
|SHP0025784 |2    |
|SHP0005611 |2    |
|SHP0011388 |2    |
|SHP0017357 |2    |
|SHP0035268 |2    |
|SHP0002969 |2    |
+-----------+-----+
only showing top 20 rows
Rows before deduplication: 87000
Rows after deduplication: 86250


In [24]:
# 20. Inspect the nested sensor JSON schema first.
sensor_df.printSchema()
sensor_df.show(truncate=False)


root
 |-- environment: struct (nullable = true)
 |    |-- humidity_target_pct: long (nullable = true)
 |    |-- temperature_range_c: struct (nullable = true)
 |    |    |-- maximum: long (nullable = true)
 |    |    |-- minimum: long (nullable = true)
 |-- safety: struct (nullable = true)
 |    |-- fire_suppression: string (nullable = true)
 |    |-- inspections: struct (nullable = true)
 |    |    |-- last_date: string (nullable = true)
 |    |    |-- next_date: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- enabled: boolean (nullable = true)
 |    |    |-- sensor_id: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- warehouse_id: string (nullable = true)

+--------------+-------------------------------------+------------------------------------------------------------------------------+------------+
|environment   |safety                               |sensors                     

In [27]:
sensor_rows_df = sensor_df.select(
    "warehouse_id",
    F.col("environment.temperature_range_c.minimum").alias("min_temperature"),
    F.col("environment.temperature_range_c.maximum").alias("max_temperature"),
    F.explode("sensors").alias("sensor_details")
)
sensor_rows_df.show(truncate=False)


+------------+---------------+---------------+----------------------------+
|warehouse_id|min_temperature|max_temperature|sensor_details              |
+------------+---------------+---------------+----------------------------+
|W01         |18             |32             |{false, W01-S1, Temperature}|
|W01         |18             |32             |{true, W01-S2, Humidity}    |
|W01         |18             |32             |{true, W01-S3, Door}        |
|W02         |18             |32             |{true, W02-S1, Temperature} |
|W02         |18             |32             |{true, W02-S2, Humidity}    |
|W02         |18             |32             |{true, W02-S3, Door}        |
|W03         |18             |32             |{true, W03-S1, Temperature} |
|W03         |18             |32             |{true, W03-S2, Humidity}    |
|W03         |18             |32             |{true, W03-S3, Door}        |
|W04         |18             |32             |{true, W04-S1, Temperature} |
|W04        

**Part G | Lazy Execution, Caching, and Persistence**

In [28]:
# 21. Build a pipeline and inspect its plan before an action.
pipeline_df = shipments_df.filter(F.col("shipment_status") == "Delivered").join(
    warehouse_df, on="warehouse_id", how="left"
).withColumn(
    "shipping_profit", F.col("shipping_fee_inr") - F.col("shipping_cost_inr")
).groupBy("region").agg(
    F.sum("shipping_fee_inr").alias("total_revenue"),
    F.sum("shipping_profit").alias("total_profit")
).orderBy(F.col("total_revenue").desc())
pipeline_df.explain()
pipeline_df.show(truncate=False)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_revenue#1410 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_revenue#1410 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1787]
      +- HashAggregate(keys=[region#94], functions=[sum(shipping_fee_inr#28), sum(shipping_profit#1409)])
         +- Exchange hashpartitioning(region#94, 200), ENSURE_REQUIREMENTS, [plan_id=1784]
            +- HashAggregate(keys=[region#94], functions=[partial_sum(shipping_fee_inr#28), partial_sum(shipping_profit#1409)])
               +- Project [shipping_fee_inr#28, region#94, (shipping_fee_inr#28 - shipping_cost_inr#29) AS shipping_profit#1409]
                  +- BroadcastHashJoin [warehouse_id#22], [warehouse_id#91], LeftOuter, BuildRight, false
                     :- Project [warehouse_id#22, shipping_fee_inr#28, shipping_cost_inr#29]
                     :  +- Filter (isnotnull(shipment_status#30) AND (shipment_status#30 = Delivered))
                     : 

In [29]:
# 22. Cache delivered shipments and compare repeated count and aggregation actions.
delivered_cached_df = shipments_df.filter(F.col("shipment_status") == "Delivered").cache()
start = time.perf_counter()
print("Delivered count:", delivered_cached_df.count())
time1 = time.perf_counter() - start
start = time.perf_counter()
print("Delivered count again:", delivered_cached_df.count())
time2 = time.perf_counter() - start
start = time.perf_counter()
delivered_cached_df.agg(F.sum("shipping_fee_inr").alias("total_fees")).show()
time3 = time.perf_counter() - start
print("First count:", round(time1, 3), "seconds")
print("Second count:", round(time2, 3), "seconds")
print("Aggregation:", round(time3, 3), "seconds")
delivered_cached_df.unpersist()


Delivered count: 52434
Delivered count again: 52434
+--------------------+
|          total_fees|
+--------------------+
|1.9528933499999907E7|
+--------------------+

First count: 2.566 seconds
Second count: 0.156 seconds
Aggregation: 0.293 seconds


DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

In [30]:
# 23. Persist Express shipments with MEMORY_AND_DISK, run two actions, and release the data.
express_df = shipments_df.filter(F.col("service_level") == "Express").persist(StorageLevel.MEMORY_AND_DISK)
print("Express count:", express_df.count())
express_df.show(5, truncate=False)
print("Storage level:", express_df.storageLevel)
express_df.unpersist()


Express count: 21090
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0000001 |2026-05-24  |2026-05-24   |202

DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

**Part H | Partitioning and Optimization**

In [31]:
# 24. Repartition shipments into 8 partitions and count records in each partition.
shipments_8_df = shipments_df.repartition(8)
partition_counts_df = shipments_8_df.withColumn(
    "partition_id", F.spark_partition_id()
).groupBy("partition_id").count().orderBy("partition_id")
partition_counts_df.show()


+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 9375|
|           1| 9374|
|           2| 9376|
|           3| 9375|
|           4| 9375|
|           5| 9375|
|           6| 9375|
|           7| 9375|
+------------+-----+



In [32]:
# 25. Repartition into 12 partitions, then reduce to 3 with coalesce(). Compare with repartition(3).
shipments_12_df = shipments_df.repartition(12)
shipments_coalesce_3_df = shipments_12_df.coalesce(3)
shipments_repartition_3_df = shipments_12_df.repartition(3)
print("12 partitions:", shipments_12_df.rdd.getNumPartitions())
print("After coalesce:", shipments_coalesce_3_df.rdd.getNumPartitions())
print("After repartition:", shipments_repartition_3_df.rdd.getNumPartitions())
print("Coalesce plan:")
shipments_coalesce_3_df.explain()
print("Repartition plan:")
shipments_repartition_3_df.explain()


12 partitions: 12
After coalesce: 3
After repartition: 3
Coalesce plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 3
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=2278]
            +- FileScan csv [shipment_id#17,booking_date#18,dispatch_date#19,promised_delivery_date#20,actual_delivery_date#21,warehouse_id#22,carrier_id#23,route_id#24,customer_segment#25,service_level#26,package_weight_kg#27,shipping_fee_inr#28,shipping_cost_inr#29,shipment_status#30,delivery_attempts#31,damage_flag#32,customer_rating#33,handling_minutes#34,payment_mode#35,priority_flag#36] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/shipments_75000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<shipment_id:string,booking_date:date,dispatch_date:date,promised_delivery_date:date,actual...
+- == Initial Plan ==
   Coalesc

In [33]:
# 26. Join shipments with warehouse data and write Parquet using 4 output partitions.
shipment_warehouse_df = shipments_df.join(warehouse_df, on="warehouse_id", how="left")
shipment_warehouse_df.repartition(4).write.mode("overwrite").parquet("output/shipment_warehouse")
print("Parquet output written to output/shipment_warehouse")


Parquet output written to output/shipment_warehouse


**Final Challenge | Logistics Performance Pipeline**

In [34]:
# 1. Join shipment, warehouse, carrier, and route data.
logistics_df = shipments_df.join(warehouse_df, on="warehouse_id", how="left").join(
    carrier_df, on="carrier_id", how="left"
).join(route_df, on="route_id", how="left")
logistics_df.printSchema()


root
 |-- route_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)
 |-- warehouse_name: string (nullable = true)
 |-- city: string (nullable = 

In [35]:
# 2. Clean missing costs and weights. Missing cost is set to zero for profit calculation;
logistics_clean_df = logistics_df.fillna({"shipping_cost_inr": 0}) \
    .withColumn("booking_date", F.to_date("booking_date")) \
    .withColumn("promised_delivery_date", F.to_date("promised_delivery_date")) \
    .withColumn("actual_delivery_date", F.to_date("actual_delivery_date"))

logistics_clean_df = logistics_clean_df.withColumn(
    "delivery_duration_days", F.datediff("actual_delivery_date", "booking_date")
).withColumn(
    "late_delivery", F.when(
        F.col("actual_delivery_date").isNotNull() &
        (F.col("actual_delivery_date") > F.col("promised_delivery_date")), 1
    ).otherwise(0)
).withColumn(
    "shipping_profit", F.col("shipping_fee_inr") - F.col("shipping_cost_inr")
)


In [36]:
# 3. Aggregate revenue, profit, and late-delivery percentage by region and carrier.
performance_summary_df = logistics_clean_df.groupBy("region", "carrier_id", "carrier_name").agg(
    F.sum("shipping_fee_inr").alias("total_revenue"),
    F.sum("shipping_profit").alias("total_profit"),
    (F.avg("late_delivery") * 100).alias("late_delivery_pct"),
    (F.avg(F.when(
        F.col("actual_delivery_date").isNotNull() &
        (F.col("actual_delivery_date") <= F.col("promised_delivery_date")), 1
    ).otherwise(0)) * 100).alias("on_time_delivery_pct")
)


In [37]:
# 4. Rank carriers within each region by on-time delivery percentage.
rank_window = Window.partitionBy("region").orderBy(F.col("on_time_delivery_pct").desc())
final_report_df = performance_summary_df.withColumn(
    "carrier_rank", F.dense_rank().over(rank_window)
)


In [38]:
# 5. Cache the final report and reuse it for multiple actions.
final_report_df.cache()
final_report_df.show(truncate=False)
print("Report rows:", final_report_df.count())
final_report_df.orderBy("region", "carrier_rank").show(truncate=False)


+------+----------+--------------------+------------------+------------------+------------------+--------------------+------------+
|region|carrier_id|carrier_name        |total_revenue     |total_profit      |late_delivery_pct |on_time_delivery_pct|carrier_rank|
+------+----------+--------------------+------------------+------------------+------------------+--------------------+------------+
|South |C05       |CargoNest           |1437480.9200000009|364710.4200000005 |28.243877019280877|43.14747264200104   |1           |
|South |C03       |MetroRoute Logistics|1929008.1199999978|494540.6199999999 |29.70944775832211 |41.5047142582259    |2           |
|South |C06       |RouteWise           |1233214.2399999998|316221.74         |29.657451923076923|41.25600961538461   |3           |
|South |C01       |SwiftLine Express   |2271284.540000001 |574166.9399999983 |28.378600823045268|40.8724279835391    |4           |
|South |C02       |ParcelBridge        |2442652.5400000014|633253.4399999988

In [39]:
# 6. Save the final report as partitioned Parquet files.
# Partitioning by region creates a folder for each region; repartition(4) controls write parallelism.
final_report_df.repartition(4).write.mode("overwrite").partitionBy("region").parquet("output/logistics_performance_report")
print("Final report written to output/logistics_performance_report")
final_report_df.unpersist()


Final report written to output/logistics_performance_report


DataFrame[region: string, carrier_id: string, carrier_name: string, total_revenue: double, total_profit: double, late_delivery_pct: double, on_time_delivery_pct: double, carrier_rank: int]